# T-205 — Benchmark de embeddings y decisión

Decide qué modelo de embeddings usa el índice semántico del catálogo (RF-301…304). Decisión **pendiente** registrada en `specs/001-cuestion-de-datos-v2/research.md` §1: bloquea `data-model.md` §`catalog_embeddings` (`<DIM>`), la migración T-104B, `EMBEDDING_MODEL` en `.env.example`/`quickstart.md`, y las dependencias definitivas de runtime en `backend/pyproject.toml`.

**Instrucciones de ejecución paso a paso: `notebooks/README.md`.** Este notebook asume que ya corriste:
1. `pip install -e ".[dev,benchmark-embeddings]"` (desde `backend/`)
2. `python scripts/export_benchmark_fixture.py` (congela la muestra en `notebooks/fixtures/`)
3. Tienes `GOOGLE_API_KEY` en el entorno (candidato gestionado)

**Candidatos comparados** (`research.md` §1): `intfloat/multilingual-e5-large` (local, 1024 dim) vs. `gemini-embedding-2` (gestionado, dimensión configurable 128–3072 vía Matryoshka Representation Learning).

**Los 10 criterios de research.md §1** que esta libreta debe dejar completos en la tabla comparativa final: (1) recall@10 en español, (2) calidad territorial colombiana, (3) dimensión del vector, (4) latencia p95, (5) costo, (6) viabilidad de ejecución local en el servidor del piloto, (7) dependencia de proveedor, (8) reproducibilidad, (9) tamaño del índice resultante, (10) compatibilidad pgvector (`DIM <= 2000` para `vector`, o `halfvec` si no).

**Lo que este notebook NO puede hacer por ti:** anotar a mano el dataset esperado de cada consulta de prueba (Sección 2) y firmar la decisión final (Sección 7) — ambos exigen juicio humano de dominio, no son automatizables sin inventar evidencia (Constitución Art. I).

## 0. Entorno — registrar versiones y hardware

`research.md` §1 exige registrar: versión de Python, versiones de librerías, CPU/RAM/dispositivo usado. Esta celda lo hace en frío antes de correr nada, para que quede pegado al reporte final sin transcripción manual.

In [ ]:
import importlib.metadata as importlib_metadata
import platform

import psutil


def registrar_entorno() -> dict:
    info = {
        "python_version": platform.python_version(),
        "platform": platform.platform(),
        "cpu_count_logical": psutil.cpu_count(logical=True),
        "cpu_count_physical": psutil.cpu_count(logical=False),
        "ram_total_gb": round(psutil.virtual_memory().total / (1024**3), 2),
    }
    for pkg in [
        "sentence-transformers",
        "torch",
        "transformers",
        "huggingface-hub",
        "pandas",
        "httpx",
        "numpy",
    ]:
        try:
            info[f"{pkg}_version"] = importlib_metadata.version(pkg)
        except importlib_metadata.PackageNotFoundError:
            info[f"{pkg}_version"] = None
    try:
        import torch

        info["torch_cuda_available"] = torch.cuda.is_available()
        info["torch_device_used"] = "cuda" if torch.cuda.is_available() else "cpu"
    except ImportError:
        info["torch_cuda_available"] = None
        info["torch_device_used"] = None
    return info


environment_info = registrar_entorno()
environment_info

## 1. Cargar la muestra congelada

Lee `notebooks/fixtures/catalog_sample.json` (metadatos de T-201, `embedding_text` ya construido según plan.md §5.4) y `notebooks/fixtures/divipola_master.json` (T-202). Si no existen, corre `python scripts/export_benchmark_fixture.py` desde `backend/` primero (`notebooks/README.md`).

In [ ]:
import json
from pathlib import Path

FIXTURES_DIR = Path("fixtures")

catalog_fixture = json.loads((FIXTURES_DIR / "catalog_sample.json").read_text(encoding="utf-8"))
divipola_fixture = json.loads((FIXTURES_DIR / "divipola_master.json").read_text(encoding="utf-8"))

corpus = [d for d in catalog_fixture["datasets"] if d.get("embedding_text")]
corpus_ids = [d["id"] for d in corpus]
corpus_texts = [d["embedding_text"] for d in corpus]

print(f"Corpus: {len(corpus)} datasets (exportado {catalog_fixture['exported_at']})")
print(f"Maestro DIVIPOLA: {divipola_fixture['count']} entradas (exportado {divipola_fixture['exported_at']})")

## 2. Consultas de prueba (golden queries) — REQUIERE TRABAJO HUMANO

`research.md` §1, procedimiento (c): **≥ 30 consultas en español, con dataset esperado anotado a mano, incluidas ≥ 10 territoriales** (usando el maestro DIVIPOLA de T-202: municipios/departamentos reales, no inventados).

**Cómo anotar sin adivinar:** usa `buscar_por_palabra_clave` de la celda siguiente para encontrar candidatos por texto en la muestra congelada, confirma a ojo cuál `dataset_id` responde realmente la pregunta (abre el dataset en datos.gov.co si hace falta) y solo entonces agrégalo a `golden_queries`. Si ningún dataset de la muestra responde una pregunta territorial razonable, es una señal legítima para el criterio 1/2 (recall bajo), no un motivo para forzar un match falso.

Las dos entradas de abajo son **EJEMPLO — bórralas o reemplázalas**; no están verificadas.

In [ ]:
def buscar_por_palabra_clave(palabra: str, limite: int = 10) -> list[tuple[str, str]]:
    """Ayuda para anotar a mano: NO decide el dataset esperado, solo acorta la
    busqueda manual en la muestra congelada (name/embedding_text)."""
    palabra_norm = palabra.lower()
    return [
        (d["id"], d["name"])
        for d in corpus
        if palabra_norm in (d["name"] or "").lower()
        or palabra_norm in (d["embedding_text"] or "").lower()
    ][:limite]


# EJEMPLO -- reemplazar por consultas reales anotadas a mano.
golden_queries: list[dict] = [
    {
        "query": "EJEMPLO: deserción escolar en Colombia",
        "expected_dataset_ids": ["REEMPLAZAR-CON-ID-REAL"],
        "territorial": False,
        "notes": "placeholder, no verificado -- borrar antes de correr el benchmark real",
    },
    {
        "query": "EJEMPLO: programas de educación en El Carmen de Viboral, Antioquia (05148)",
        "expected_dataset_ids": ["REEMPLAZAR-CON-ID-REAL"],
        "territorial": True,
        "notes": "placeholder, no verificado -- borrar antes de correr el benchmark real",
    },
]

assert len(golden_queries) >= 30, "research.md §1: se requieren >= 30 consultas anotadas a mano"
assert sum(q["territorial"] for q in golden_queries) >= 10, "se requieren >= 10 consultas territoriales"

## 3. Candidato local — `intfloat/multilingual-e5-large`

Los modelos E5 son **asimétricos**: exigen el prefijo `"query: "` para consultas y `"passage: "` para documentos del corpus (convención documentada en la model card de `intfloat/multilingual-e5-large`). Omitir el prefijo degrada notablemente la calidad de recuperación — no es un detalle cosmético.

In [ ]:
import time

from sentence_transformers import SentenceTransformer

E5_MODEL_ID = "intfloat/multilingual-e5-large"


def load_local_model(model_id: str = E5_MODEL_ID) -> tuple[SentenceTransformer, float]:
    t0 = time.perf_counter()
    model = SentenceTransformer(model_id)
    load_time_s = time.perf_counter() - t0
    return model, load_time_s


def embed_local(
    model: SentenceTransformer, texts: list[str], prefix: str, batch_size: int = 16
) -> tuple[list[list[float]], list[float]]:
    """prefix: 'query: ' o 'passage: '. Devuelve (vectores, latencia_ms_por_texto_por_lote)."""
    prefixed = [f"{prefix}{t}" for t in texts]
    latencies_ms: list[float] = []
    vectors: list[list[float]] = []
    for i in range(0, len(prefixed), batch_size):
        batch = prefixed[i : i + batch_size]
        t0 = time.perf_counter()
        batch_vectors = model.encode(batch, normalize_embeddings=True, show_progress_bar=False)
        elapsed_ms = (time.perf_counter() - t0) * 1000
        latencies_ms.extend([elapsed_ms / len(batch)] * len(batch))
        vectors.extend(v.tolist() for v in batch_vectors)
    return vectors, latencies_ms


# Descomentar para correr el candidato local (descarga ~1-2 GB de pesos la primera vez):
# local_model, local_load_time_s = load_local_model()
# corpus_vectors_local, corpus_latencies_local = embed_local(local_model, corpus_texts, prefix="passage: ")
# query_vectors_local, query_latencies_local = embed_local(
#     local_model, [q["query"] for q in golden_queries], prefix="query: "
# )

## 4. Candidato gestionado — `gemini-embedding-2`

Llamada REST directa (`embedContentConfig.taskType` / `outputDimensionality`, verificado en `https://ai.google.dev/api/embeddings` el 2026-07-09) en vez del wrapper `langchain-google-genai` instalado en el proyecto: la versión actual de ese wrapper no expone `output_dimensionality`, y el criterio 3/9/10 de research.md §1 exige comparar dimensiones distintas del mismo modelo (128–3072, Matryoshka). Precio verificado el 2026-07-09 en `https://ai.google.dev/gemini-api/docs/pricing`: USD 0,20 / 1M tokens de entrada (tier pagado); tier gratuito sin costo pero con límites de tasa más bajos — **vuelve a verificar el precio vigente antes de usarlo para una decisión de presupuesto real**, los proveedores cambian precios sin aviso.

In [ ]:
import os

import httpx

GEMINI_MODEL_ID = "gemini-embedding-2"
GEMINI_API_BASE = "https://generativelanguage.googleapis.com/v1beta"
GEMINI_BATCH_MAX = 100  # limite documentado de batchEmbedContents; confirmar si Google lo cambia
GEMINI_PRICE_PER_1M_TOKENS_USD = 0.20  # tier pagado, verificar vigencia antes de decidir


def _gemini_headers() -> dict[str, str]:
    return {"x-goog-api-key": os.environ["GOOGLE_API_KEY"], "Content-Type": "application/json"}


def embed_gemini(
    texts: list[str],
    task_type: str,
    output_dimensionality: int,
    model_id: str = GEMINI_MODEL_ID,
    batch_size: int = GEMINI_BATCH_MAX,
) -> tuple[list[list[float]], list[float], int]:
    """task_type: 'RETRIEVAL_QUERY' o 'RETRIEVAL_DOCUMENT'.
    Devuelve (vectores, latencia_ms_por_texto_por_lote, tokens_totales_reales)."""
    url = f"{GEMINI_API_BASE}/models/{model_id}:batchEmbedContents"
    vectors: list[list[float]] = []
    latencies_ms: list[float] = []
    total_tokens = 0
    with httpx.Client(timeout=30.0) as client:
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            body = {
                "requests": [
                    {
                        "model": f"models/{model_id}",
                        "content": {"parts": [{"text": text}]},
                        "embedContentConfig": {
                            "taskType": task_type,
                            "outputDimensionality": output_dimensionality,
                        },
                    }
                    for text in batch
                ]
            }
            t0 = time.perf_counter()
            response = client.post(url, headers=_gemini_headers(), json=body)
            response.raise_for_status()
            elapsed_ms = (time.perf_counter() - t0) * 1000
            payload = response.json()
            latencies_ms.extend([elapsed_ms / len(batch)] * len(batch))
            vectors.extend(item["values"] for item in payload["embeddings"])
            total_tokens += payload.get("usageMetadata", {}).get("promptTokenCount", 0)
    return vectors, latencies_ms, total_tokens


def costo_estimado_usd(n_tokens: int, precio_por_millon: float = GEMINI_PRICE_PER_1M_TOKENS_USD) -> float:
    return (n_tokens / 1_000_000) * precio_por_millon


# Descomentar para correr el candidato gestionado (requiere GOOGLE_API_KEY en el entorno).
# GEMINI_OUTPUT_DIM = 768  # <= 2000: compatible con vector(<DIM>) sin pasar a halfvec (research.md §1)
# corpus_vectors_gemini, corpus_latencies_gemini, corpus_tokens_gemini = embed_gemini(
#     corpus_texts, task_type="RETRIEVAL_DOCUMENT", output_dimensionality=GEMINI_OUTPUT_DIM
# )
# query_vectors_gemini, query_latencies_gemini, query_tokens_gemini = embed_gemini(
#     [q["query"] for q in golden_queries], task_type="RETRIEVAL_QUERY", output_dimensionality=GEMINI_OUTPUT_DIM
# )

## 5. Recall@10 y latencia p50/p95

Búsqueda por similitud coseno pura en memoria (sin pgvector): el objetivo de este notebook es comparar candidatos antes de decidir, no operar el índice real.

In [ ]:
import numpy as np


def cosine_topk(query_vec: list[float], corpus_vecs: list[list[float]], k: int = 10) -> list[int]:
    q = np.array(query_vec)
    c = np.array(corpus_vecs)
    q = q / np.linalg.norm(q)
    c = c / np.linalg.norm(c, axis=1, keepdims=True)
    scores = c @ q
    return list(np.argsort(-scores)[:k])


def recall_at_k(
    queries: list[dict],
    query_vectors: list[list[float]],
    ids: list[str],
    corpus_vectors: list[list[float]],
    k: int = 10,
) -> float:
    hits = 0
    for q, qvec in zip(queries, query_vectors, strict=True):
        top_idx = cosine_topk(qvec, corpus_vectors, k=k)
        retrieved_ids = {ids[i] for i in top_idx}
        if retrieved_ids & set(q["expected_dataset_ids"]):
            hits += 1
    return hits / len(queries)


def recall_at_k_territorial(
    queries: list[dict],
    query_vectors: list[list[float]],
    ids: list[str],
    corpus_vectors: list[list[float]],
    k: int = 10,
) -> float:
    idx_territorial = [i for i, q in enumerate(queries) if q["territorial"]]
    subset_queries = [queries[i] for i in idx_territorial]
    subset_vectors = [query_vectors[i] for i in idx_territorial]
    return recall_at_k(subset_queries, subset_vectors, ids, corpus_vectors, k=k)


def p50_p95(latencies_ms: list[float]) -> tuple[float, float]:
    arr = np.array(latencies_ms)
    return float(np.percentile(arr, 50)), float(np.percentile(arr, 95))


# Ejemplo de uso una vez corridas las celdas 3 y 4 (descomentadas):
# recall_local = recall_at_k(golden_queries, query_vectors_local, corpus_ids, corpus_vectors_local)
# recall_local_territorial = recall_at_k_territorial(golden_queries, query_vectors_local, corpus_ids, corpus_vectors_local)
# recall_gemini = recall_at_k(golden_queries, query_vectors_gemini, corpus_ids, corpus_vectors_gemini)
# recall_gemini_territorial = recall_at_k_territorial(golden_queries, query_vectors_gemini, corpus_ids, corpus_vectors_gemini)

## 6. Tamaño del índice y tabla comparativa (10 criterios de research.md §1)

Tamaño estimado del índice: `n_datasets_totales_del_catalogo × dimensión × 4 bytes` (float32). Usa el conteo real del catálogo completo (T-201, ~8.400 datasets a la fecha de este PR — confirma con `SELECT count(*) FROM catalog_datasets WHERE api_active;`), no el tamaño de la muestra congelada.

In [ ]:
import pandas as pd

CATALOG_TOTAL_DATASETS_ACTIVE = None  # TODO: pegar el conteo real de `SELECT count(*) FROM catalog_datasets WHERE api_active;`


def tamano_indice_mb(n_datasets: int, dim: int) -> float:
    return round((n_datasets * dim * 4) / (1024**2), 2)


# Rellenar con los resultados reales de las secciones 3-5 antes de decidir.
comparacion = pd.DataFrame(
    {
        "criterio": [
            "1. recall@10 (todas las consultas)",
            "2. recall@10 (solo territoriales)",
            "3. dimensión del vector",
            "4. latencia p95 embed consulta (ms)",
            "5. costo estimado / 1.000 consultas (USD)",
            "5. costo estimado reindexación completa (USD)",
            "6. ¿corre en el servidor del piloto? (RAM/CPU medidos arriba vs. tier Railway/Render)",
            "7. dependencia de proveedor (lock-in)",
            "8. reproducibilidad (¿un tercero regenera el índice idéntico?)",
            "9. tamaño del índice resultante (MB, catálogo completo)",
            "10. compatibilidad pgvector (vector <=2000 / halfvec)",
        ],
        "intfloat/multilingual-e5-large (local)": [None] * 11,
        "gemini-embedding-2 (gestionado)": [None] * 11,
    }
)
comparacion

## 7. Decisión razonada — A COMPLETAR Y FIRMAR POR EL RESPONSABLE DEL PROYECTO

`research.md` §1, procedimiento (f): *"decisión razonada firmada por el responsable del proyecto"*. Esta celda es una plantilla vacía a propósito — la decisión y su justificación las escribe Camilo con los resultados reales de arriba, no este notebook.

**Modelo elegido:** _(pendiente)_

**Dimensión elegida:** _(pendiente)_

**Tipo de columna pgvector (`vector` / `halfvec`):** _(pendiente — `halfvec` obligatorio si `DIM > 2000`)_

**Justificación (referencia explícita a los 10 criterios medidos arriba):** _(pendiente)_

**Firma y fecha:** _(pendiente)_

---

**Después de firmar, propaga la decisión en el mismo PR (`notebooks/README.md` tiene el detalle):**
- [ ] `specs/001-cuestion-de-datos-v2/research.md` §1 — decisión + evidencia
- [ ] `specs/001-cuestion-de-datos-v2/data-model.md` — reemplazar `<DIM>` en `catalog_embeddings`
- [ ] `specs/001-cuestion-de-datos-v2/plan.md` — stack de embeddings, ya no "DECISIÓN PENDIENTE"
- [ ] `backend/.env.example` y `quickstart.md` — `EMBEDDING_MODEL`
- [ ] `backend/pyproject.toml` — dependencia definitiva de runtime (fuera de `benchmark-embeddings`)
- [ ] T-104B queda desbloqueada en `tasks.md`